<a href="https://colab.research.google.com/github/Danny3636/Generative-AI-Tasks/blob/main/Required_Task_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

From this task, I learned how to set up and use the tabpfn-client library by cloning its repository and installing its dependencies. I then applied TabPFN to a classification problem, comparing its performance against Random Forest and XGBoost, and observed that TabPFN yielded slightly higher accuracy and ROC-AUC scores on this dataset.


In [ ]:
!pip install tabpfn


## Clone tabpfn-client repository



In [ ]:
import subprocess

# Clone the tabpfn-client repository
repo_url = "https://github.com/PriorLabs/tabpfn-client.git"
subprocess.run(["git", "clone", repo_url])

print(f"Repository {repo_url} cloned successfully.")

Repository https://github.com/PriorLabs/tabpfn-client.git cloned successfully.


In [ ]:
import os

# Change current directory to tabpfn-client
os.chdir('tabpfn-client')
print(f"Current working directory: {os.getcwd()}")

# List contents of the directory to check for requirements.txt
print("Contents of tabpfn-client directory:")
print(os.listdir('.'))

Current working directory: /content/tabpfn-client/tabpfn-client
Contents of tabpfn-client directory:
['tests', 'src', 'scripts', 'README.md', '.git', 'uv.lock', '.pre-commit-config.yaml', 'pyproject.toml', '.github', '.gitignore', 'LICENSE', 'ruff.toml']


In [ ]:
import subprocess

# Install the tabpfn-client package and its dependencies
subprocess.run(["pip", "install", "."])

print("tabpfn-client and its dependencies installed successfully.")

tabpfn-client and its dependencies installed successfully.


In [ ]:
import os

# List contents of the scripts directory to find examples
print("Contents of tabpfn-client/scripts directory:")
print(os.listdir('scripts'))

Contents of tabpfn-client/scripts directory:
['gen_requirements.py']


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from tabpfn_client import TabPFNClassifier # Changed import to directly from tabpfn_client
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# 1. Load Dataset (UCI Credit Card Default)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls"
df = pd.read_excel(url, header=1)

# Features and Target
X = df.drop(columns=['ID', 'default payment next month'])
y = df['default payment next month']

# 2. Split Data (25% Test Set)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# 3. Initialize Models
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
# Updated TabPFNClassifier initialization for tabpfn_client, removed 'device' and 'N_ensemble_configurations' arguments
tabpfn = TabPFNClassifier(model_path='v2.5_default', ignore_pretraining_limits=True)

# 4. Train and Predict
models = {"Random Forest": rf_model, "XGBoost": xgb_model, "TabPFN": tabpfn}
results = {}

for name, model in models.items():
    if name == "TabPFN":
        # Removed data subsetting for TabPFN
        model.fit(X_train, y_train)
    else:
        model.fit(X_train, y_train)

    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    results[name] = {
        "Accuracy": accuracy_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, probs)
    }

# 5. Display Comparison
print(pd.DataFrame(results).T)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [15:30:28] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


00:00 Fitting... Done!


Processing: 100%|██████████| [00:10<00:00]
Processing: 100%|██████████| [00:05<00:00]

               Accuracy   ROC-AUC
Random Forest  0.816000  0.753386
XGBoost        0.812667  0.759529
TabPFN         0.821467  0.783043
